In [2]:
# Install required packages if not already installed
# !pip install sentence-transformers textblob nltk

from sentence_transformers import SentenceTransformer, util
from textblob import TextBlob
from nltk.translate.bleu_score import sentence_bleu

# Load embedding model
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

def evaluate_responses(response1, response2):
    """
    Evaluate similarity and sentiment alignment between two responses.
    response1: str, first response (e.g., model-generated)
    response2: str, second response (e.g., reference or another model)
    """

    # --- 1. Semantic similarity ---
    embeddings = embed_model.encode([response1, response2], convert_to_tensor=True)
    cosine_sim = util.cos_sim(embeddings[0], embeddings[1]).item()

    # --- 2. Sentiment analysis ---
    sentiment1 = TextBlob(response1).sentiment.polarity  # range [-1,1]
    sentiment2 = TextBlob(response2).sentiment.polarity
    sentiment_diff = abs(sentiment1 - sentiment2)

    # --- 3. BLEU score (optional) ---
    reference = [response2.split()]
    candidate = response1.split()
    bleu_score = sentence_bleu(reference, candidate)

    # --- 4. Print results ---
    print("Semantic similarity (cosine):", round(cosine_sim, 3))
    print("Sentiment polarity difference:", round(sentiment_diff, 3))
    print("BLEU score:", round(bleu_score, 3))

    return {
        "semantic_similarity": cosine_sim,
        "sentiment_difference": sentiment_diff,
        "bleu_score": bleu_score
    }

# --- Example usage ---
resp1 = "I see that you're feeling sad lately, is there anything specific that's been bothering you? It can be helpful to talk about it if you feel comfortable doing so. Would you like me to listen or offer any advice on how to cope with your current situation?"
resp2 = "Have you considered seeking support from professionals or support groups? They may be able to provide you with additional resources and coping strategies."

results = evaluate_responses(resp1, resp2)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Semantic similarity (cosine): 0.392
Sentiment polarity difference: 0.58
BLEU score: 0.0


/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_

In [3]:
# Install packages if not already
# !pip install sentence-transformers textblob nltk

from sentence_transformers import SentenceTransformer, util
from textblob import TextBlob
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Load embedding model
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

def evaluate_responses(response1, response2):
    """
    Evaluate two responses and indicate which one is better (closer to response2).
    response1: str, candidate response
    response2: str, reference response
    """
    # --- 1. Semantic similarity ---
    embeddings = embed_model.encode([response1, response2], convert_to_tensor=True)
    cosine_sim = util.cos_sim(embeddings[0], embeddings[1]).item()

    # --- 2. Sentiment analysis ---
    sentiment1 = TextBlob(response1).sentiment.polarity
    sentiment2 = TextBlob(response2).sentiment.polarity
    sentiment_diff = abs(sentiment1 - sentiment2)

    # --- 3. Smoothed BLEU score ---
    smoothie = SmoothingFunction().method1
    reference = [response2.split()]
    candidate = response1.split()
    bleu_score = sentence_bleu(reference, candidate, smoothing_function=smoothie)

    # --- 4. Combined score (simple) ---
    # Higher cosine sim, higher BLEU, lower sentiment_diff => better
    combined_score = cosine_sim * 0.5 + bleu_score * 0.3 + (1 - sentiment_diff) * 0.2

    # --- 5. Decide which is better ---
    better = "response1" if combined_score >= 0.5 else "response2"

    # --- 6. Print metrics ---
    print("Semantic similarity (cosine):", round(cosine_sim, 3))
    print("Sentiment polarity difference:", round(sentiment_diff, 3))
    print("Smoothed BLEU score:", round(bleu_score, 3))
    print("Combined score (0-1):", round(combined_score, 3))
    print(f"Better response (closer to reference): {better}")

    return {
        "cosine_similarity": cosine_sim,
        "sentiment_difference": sentiment_diff,
        "bleu_score": bleu_score,
        "combined_score": combined_score,
        "better_response": better
    }

# --- Example usage ---
resp1 = "I see that you're feeling sad lately, is there anything specific that's been bothering you? It can be helpful to talk about it if you feel comfortable doing so. Would you like me to listen or offer any advice on how to cope with your current situation?"
resp2 = "Have you considered seeking support from professionals or support groups? They may be able to provide you with additional resources and coping strategies."


results = evaluate_responses(resp1, resp2)


Semantic similarity (cosine): 0.392
Sentiment polarity difference: 0.58
Smoothed BLEU score: 0.006
Combined score (0-1): 0.282
Better response (closer to reference): response2
